In [1]:
pip install langchain-documentloaders langchain pypdf

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement langchain-documentloaders (from versions: none)
ERROR: No matching distribution found for langchain-documentloaders
You should consider upgrading via the 'c:\Users\LOQ\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
pip install pypdf

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\LOQ\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader(r"the_constitution_of_india.pdf")
documents = loader.load()

In [4]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 100
)

chunks = text_splitter.split_documents(documents)


In [5]:
print(chunks[10])

page_content='54. Election of President. 
55. Manner of election of President. 
56. Term of office of President. 
57. Eligibility for re-election. 
58. Qualifications for election as President. 
59. Conditions of President‘s office. 
60. Oath or affirmation by the President. 
61. Procedure for impeachment of the President. 
62. Time of holding election to fill vacancy in the office of President and the term of office of 
person elected to fill casual vacancy. 
63. The Vice-President of India. 
64. The Vice-President to be ex officio Chairman of the Council of States. 
65. The Vice-President to act as President or to discharge his functions during casual vacancies 
in the office, or during the absence, of President. 
66. Election of Vice-President. 
67. Term of office of Vice-President. 
68. Time of holding election to fill vacancy in the office of Vice-President and the term of office of 
person elected to fill casual vacancy. 
69. Oath or affirmation by the Vice-President.' metadata={

In [9]:
from langchain_ollama import OllamaEmbeddings
embeddings = OllamaEmbeddings(
    model='llama3.2'
)
try:
    vector = embeddings.embed_query("what is the first law of the constitution?")
except ConnectionError as e:
    print("Failed to connect to Ollama. Please ensure Ollama is downloaded, running, and accessible.")
    vector = None

In [10]:
vector

[-0.006068064,
 0.0048334775,
 -0.010274448,
 -0.013661343,
 -0.0014937642,
 -0.008545337,
 -0.00093387684,
 0.009132497,
 0.0030420502,
 -0.023633657,
 0.0022199918,
 0.018351885,
 0.004270088,
 -0.009538986,
 -0.011963961,
 0.012437174,
 0.010647865,
 -0.0053442055,
 -0.011888768,
 -0.015385419,
 -0.0049808654,
 -0.0070631327,
 0.022365928,
 0.016078837,
 0.03464979,
 0.0038513618,
 -0.0007109245,
 -0.0075862096,
 -0.008152751,
 -0.004145459,
 0.007671609,
 0.026581725,
 0.012457338,
 0.019667089,
 0.021043884,
 -0.009272311,
 0.009148168,
 -0.016988927,
 -0.01839672,
 0.001782935,
 -0.01704105,
 0.011126889,
 0.009429724,
 0.017819097,
 0.00995586,
 0.030736119,
 0.005766353,
 -0.024178421,
 0.009419504,
 -0.019455373,
 0.007065418,
 -0.019859098,
 -0.01029749,
 0.024578009,
 0.017377544,
 -0.009327399,
 0.0059343195,
 -0.035621975,
 -0.025331583,
 -0.0022089563,
 -0.007824578,
 -0.0012855375,
 0.0031374004,
 -0.031459585,
 -0.01297054,
 -0.009514276,
 -0.014445217,
 0.015957164,
 -

In [11]:
pip install faiss-cpu

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\LOQ\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [12]:
from langchain_community.vectorstores import FAISS
db = FAISS.from_documents(chunks, embeddings)

In [13]:
retriever = db.as_retriever(search_type="similarity", search_kwargs={"k" : 4})

In [14]:
retriever.invoke("What is right to freedom?")

[Document(id='c92fc8a4-1d9c-4272-8875-c1329d6db022', metadata={'producer': 'Microsoft® Word 2010', 'creator': 'Microsoft® Word 2010', 'creationdate': '2020-12-09T20:13:14-08:00', 'author': 'Naseem', 'moddate': '2020-12-09T20:13:14-08:00', 'source': 'the_constitution_of_india.pdf', 'total_pages': 256, 'page': 3, 'page_label': '4'}, page_content='11. Parliament to regulate the right of citizenship by law. \nPART III \nFUNDAMENTAL RIGHTS \nGeneral \n12. Definition.  \n13. Laws inconsistent with or in derogation of the fundamental rights. \nRight to Equality \n14. Equality before law. \n15. Prohibition of discrimination on grounds of religion, race, caste, sex or place of birth. \n16. Equality of opportunity in matters of public employment.  \n17. Abolition of Untouchability. \n18. Abolition of titles. \nRight to Freedom \n19. Protection of certain rights regarding freedom of speech, etc. \n20. Protection in respect of conviction for offences. \n21. Protection of life and personal liberty.

In [15]:
from langchain_ollama import ChatOllama
llm = ChatOllama(
    model='llama3.2'
)

In [16]:
from langchain_core.prompts import PromptTemplate
prompt = PromptTemplate(
    template="""
    You're a helpful assistant
    Answer only from the provided transcript context.
    If the context is insufficient, just say you don't know.

    {context}
    Question : {question}
    """,
    input_variables=['context', 'question']
)

In [17]:
question = "Tell about all rights?"
retrieved_docs = retriever.invoke(question)

In [18]:
context_text = "\n\n".join(docs.page_content for docs in retrieved_docs)
context_text

'(w.e.f. 30-5-1987).\n\n4. Naval, military and air force works. \n5. Arms, firearms, ammunition and explosives. \n6. Atomic energy and mineral resources necessary for its production. \n7. Industries declared by Parliament by law to be necessary for the purpose of defence or for the \nprosecution of war. \n8. Central Bureau of Intelligence and Investigation. \n9. Preventive detention for reasons connected with Defence, Foreign Affairs, or the security of India; \npersons subjected to such detention. \n10. Foreign affairs; all matters which bring the Union into relation with any foreign country. \n11. Diplomatic, consular and trade representation. \n12. United Nations Organisation. \n13. Participation in international conferences, associations and other bodies and implementing of \ndecisions made thereat. \n14. Entering into treaties and agreements with foreign countries and implementing of treaties, \nagreements and conventions with foreign countries. \n15. War and peace. \n16. Foreign 

In [19]:
final_prompt = prompt.invoke({
    "context" : context_text,
    "question" : question
})

In [20]:
answer = llm.invoke(final_prompt)
print(answer.content)

I don't know. The provided transcript context appears to be related to Indian law, specifically Articles 214 and others, but it does not explicitly mention "rights". It discusses various domains and industries under the jurisdiction of certain courts or authorities. If you'd like to provide more context, I can try to assist further.


In [21]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

def format_docs(retrieved_docs):
    context_text = "\n\n".join(docs.page_content for docs in retrieved_docs)
    return context_text

parallel_chain = RunnableParallel({
    'context' : retriever | RunnableLambda(format_docs),
    'question' : RunnablePassthrough()
})

parallel_chain.invoke("Tell me about Right to Equality?")

{'context': 'Advocates-on-Record Association and Another Vs. Union of India reported in AIR 2016 SC 117.\n\nAssociation and Another Vs. Union of India reported in AIR 2016 SC 117. Before Amendment Sub-clause (a) was as under:– \n ―(a) the reference in article 217 to the Governor of the State shall be construed as to the reference to the Governors of all  the \nState‘s in relation to which the High Court exercises jurisdiction.‖.\n\n1. The words ―and for the precedence of such discussion over other business of the House‖ omitted by the Constitution (First \nAmendment) Act, 1951, s. 9 (w.e.f. 18-6-1951).\n\n1. The words and letters  ―specified in Part A and Part B of the First Schedule‖ omitted by the Co nstitution (Seventh Amendment) \nAct, 1956,  s.29 and Sch. (w.e.f. 1-11-1956).',
 'question': 'Tell me about Right to Equality?'}

In [23]:
main_chain = parallel_chain | prompt | llm | parser
main_chain.invoke("I am a person and I need am fighting for my equality which article should I need to know ?")

"I don't know. The context provided is related to Indian law and the Constitution, but it doesn't mention anything about equality or your personal situation."

In [24]:
result = main_chain
result.invoke("Ok tell me about right to equality?")

'Right to Equality is mentioned under Article 14 of the Constitution.\n\nAccording to Article 14, Equality before law means that all citizens are equal before the law and are entitled to equal protection of the laws without any discrimination on grounds of religion, race, caste, sex or place of birth.'